In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv) 

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os


# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

Import all the necessary libraries:

In [6]:
import os
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error
from scipy.sparse import hstack
import lightgbm as lgb

In [7]:

device = torch.device("mps" if torch.mps.is_available() else "cpu")
print("Using device:", device)

Using device: mps


Load train and test data:

In [8]:


train = pd.read_csv( 'train.csv')
test = pd.read_csv( 'test.csv')

In [9]:
for col in train.columns:
    print(col)

sample_id
catalog_content
image_link
price


Utility Functions:

In [10]:
# unit_map = {
#     "ml": "ml", "milliliter": "ml", "milliliters": "ml", "millilitres": "ml", "millilitres": "ml",
#     "l": "l", "liter": "l", "litre": "l",
#     "oz": "oz", "ounce": "oz", "ounces": "oz",
#     "fl oz": "fl_oz", "fluid ounce": "fl_oz", "fluid ounces": "fl_oz",
#     "g": "g", "gram": "g", "grams": "g", "gm": "g",
#     "kg": "kg", "kilogram": "kg", "kilograms": "kg", "kilo gram": "kg", "kilo grams": "kg",
#     "count": "count", "ct": "count", "pcs": "count", "piece": "count", "pack": "pack"
# }

# keywords = ["pack", "organic", "premium", "bundle", "eco", "vegan", "gluten", "sugar", "diet", "mix", "instant"]

# def clean_text(text):
#     """Basic text cleaning for TF-IDF"""
#     text = re.sub(r"â€“|â€|Ã|™", "", str(text))
#     text = str(text).lower()
#     text = re.sub(r'[^a-z0-9\s\.\%\-]+', ' ', text)
#     text = re.sub(r'\s+', ' ', text).strip()
#     return text

# def extract_value(text):
#     match = re.search(r"Value:\s*([\d\.]+)", str(text))
#     return float(match.group(1)) if match else np.nan

# def extract_unit(text):
#     text = str(text).lower()
#     for pattern, unit in unit_map.items():
#         if re.search(rf'\b{pattern}\b', text):
#             return unit
#     return "unknown"

# def strip_item_name(text):
#     return re.sub(r'\bitem name\b', '', str(text), flags=re.IGNORECASE).strip()

# def feature_engineering(df):
#     """Feature engineering from catalog content"""
#     df["catalog_content"] = df["catalog_content"].fillna("").apply(clean_text)
#     df["Value"] = df["catalog_content"].apply(extract_value)
#     df["Unit"] = df["catalog_content"].apply(extract_unit)
#     df["Item_Name"] = df["catalog_content"].apply(strip_item_name)

#     # Fill missing numeric features
#     df["Value"] = df["Value"].fillna(df["Value"].median())
#     df["Unit"] = df["Unit"].fillna("unknown")



#     df["Value"] = pd.to_numeric(df["Value"], errors="coerce").fillna(0)
#     df["Value"] = df["Value"].clip(lower=0)
#     df["log_value"] = np.log1p(df["Value"])
#     df["word_count"] = df["catalog_content"].apply(lambda x: len(x.split()))
#     df["char_count"] = df["catalog_content"].apply(len)
#     df["digit_ratio"] = df["catalog_content"].apply(lambda x: sum(c.isdigit() for c in x) / max(len(x), 1))
#     df["avg_word_len"] = df["catalog_content"].apply(lambda x: np.mean([len(w) for w in x.split()]) if len(x.split()) else 0)

#     df["bullet_count"] = df["catalog_content"].str.count("bullet point")


#     # Keyword flags (you can tune/add more keywords as you analyze data)
#     for kw in keywords:
#         df[f"has_{kw}"] = df["catalog_content"].str.contains(kw, case=False).astype(int)

#     return df

In [11]:
import re
import numpy as np
import pandas as pd

# Clean text helper
def clean_text(text):
    text = re.sub(r"â€“|â€|Ã|™", "", str(text))
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s\.\%\-]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_value(text):
    match = re.search(r"Value:\s*([\d\.]+)", str(text))
    return float(match.group(1)) if match else np.nan

def extract_unit(text):
    unit_map = {
        "ml": "ml", "milliliter": "ml", "milliliters": "ml", "millilitres": "ml",
        "l": "l", "liter": "l", "litre": "l",
        "oz": "oz", "ounce": "oz", "ounces": "oz",
        "fl oz": "fl_oz", "fluid ounce": "fl_oz", "fluid ounces": "fl_oz",
        "g": "g", "gram": "g", "grams": "g", "gm": "g",
        "kg": "kg", "kilogram": "kg", "kilograms": "kg", "kilo gram": "kg", "kilo grams": "kg",
        "count": "count", "ct": "count", "pcs": "count", "piece": "count", "pack": "pack"
    }
    text = str(text).lower()
    for pattern, unit in unit_map.items():
        if re.search(rf'\b{pattern}\b', text):
            return unit
    return "unknown"

def extract_item_name(text):
    match = re.search(r"Item Name:\s*(.*?)(?=Bullet Point:|Product Description:|Value:|Unit:|$)", str(text), flags=re.DOTALL | re.IGNORECASE)
    return match.group(1).strip() if match else ""

def extract_bullet_points(text):
    # Find all bullet points and join them into one string
    bullets = re.findall(r"Bullet Point\s*\d*:\s*(.*?)(?=Bullet Point|Item Name:|Product Description:|Value:|Unit:|$)", str(text), flags=re.DOTALL | re.IGNORECASE)
    return " ".join([b.strip() for b in bullets])

def extract_product_description(text):
    match = re.search(r"Product Description:\s*(.*?)(?=Item Name:|Bullet Point:|Value:|Unit:|$)", str(text), flags=re.DOTALL | re.IGNORECASE)
    return match.group(1).strip() if match else ""

keywords = ["pack", "organic", "premium", "bundle", "eco", "vegan", "gluten", "sugar", "diet", "mix", "instant",
    "new", "fresh", "natural", "handmade", "limited", "sale", "discount", "special", "gift", "free",
    "imported", "authentic", "original", "pure", "light", "soft", "strong", "durable", "high-quality",
    "bestseller", "top-rated", "popular", "classic", "eco-friendly", "recyclable", "non-toxic", "safe",
    "baby", "kids", "adult", "men", "women", "unisex", "winter", "summer", "travel", "portable",
    "premium-grade", "luxury", "compact", "multi-purpose", "versatile", "handcrafted", "exclusive"]

def keyword_flag(text, kw):
    # Negative patterns
    neg_patterns = [r'\bno\s+', r'\bnot\s+', r'\bnon\s+', r'\bwithout\s+']
    # If any negation appears before keyword, return 0
    for neg in neg_patterns:
        if re.search(neg + kw, text, flags=re.IGNORECASE):
            return 0
    # Otherwise, if keyword exists
    return 1 if re.search(rf'\b{kw}\b', text, flags=re.IGNORECASE) else 0
def feature_engineering(df, keywords=None):
    
    keywords = ["pack", "organic", "premium", "bundle", "eco", "vegan", "gluten", "sugar", "diet", "mix", "instant",
    "new", "fresh", "natural", "handmade", "limited", "sale", "discount", "special", "gift", "free",
    "imported", "authentic", "original", "pure", "light", "soft", "strong", "durable", "high-quality",
    "bestseller", "top-rated", "popular", "classic", "eco-friendly", "recyclable", "non-toxic", "safe",
    "baby", "kids", "adult", "men", "women", "unisex", "winter", "summer", "travel", "portable",
    "premium-grade", "luxury", "compact", "multi-purpose", "versatile", "handcrafted", "exclusive"]

    
    """Feature engineering from catalog content with separate columns."""
    df["catalog_content"] = df["catalog_content"].fillna("")

    # Extract structured sections
    df["item_name_raw"] = df["catalog_content"].apply(extract_item_name)
    df["bullet_points_raw"] = df["catalog_content"].apply(extract_bullet_points)
    df["product_desc_raw"] = df["catalog_content"].apply(extract_product_description)
    df["value_raw"] = df["catalog_content"].apply(extract_value)
    df["unit_raw"] = df["catalog_content"].apply(extract_unit)

    # Clean text columns
    df["item_name"] = df["item_name_raw"].apply(clean_text)
    df["bullet_points"] = df["bullet_points_raw"].apply(clean_text)
    df["product_desc"] = df["product_desc_raw"].apply(clean_text)

    # Fill missing numeric
    df["value_raw"] = df["value_raw"].fillna(df["value_raw"].median())
    df["unit_raw"] = df["unit_raw"].fillna("unknown")

    # Numeric / derived features
    df["log_value"] = np.log1p(df["value_raw"])
    df["bullet_point_count"] = df["bullet_points_raw"].apply(lambda x: len(re.findall(r'\.', x)))  # approximate count
    df["word_count_item"] = df["item_name"].apply(lambda x: len(x.split()))
    df["word_count_bullet"] = df["bullet_points"].apply(lambda x: len(x.split()))
    df["word_count_desc"] = df["product_desc"].apply(lambda x: len(x.split()))
    df["word_count_total"] = df["word_count_item"] + df["word_count_bullet"] + df["word_count_desc"]
    df["char_count_total"] = df["item_name"].apply(len) + df["bullet_points"].apply(len) + df["product_desc"].apply(len)
    df["digit_ratio_total"] = df["item_name"].apply(lambda x: sum(c.isdigit() for c in x)/max(len(x),1)) + \
                              df["bullet_points"].apply(lambda x: sum(c.isdigit() for c in x)/max(len(x),1)) + \
                              df["product_desc"].apply(lambda x: sum(c.isdigit() for c in x)/max(len(x),1))
    df["avg_word_len_total"] = df["char_count_total"] / df["word_count_total"].replace(0,1)

    # Keyword flags
    
    for kw in keywords:
        df[f"has_{kw}"] = df["catalog_content"].apply(lambda x: keyword_flag(x, kw))

    return df

Feature Engineering:

In [12]:
train = feature_engineering(train)
test = feature_engineering(test)

# Encode Unit
le = LabelEncoder()
train["unit_enc"] = le.fit_transform(train["unit_raw"])
test["unit_enc"] = le.transform(test["unit_raw"].map(lambda x: x if x in le.classes_ else "unknown"))

TF-IDF Features:

In [13]:
# tfidf = TfidfVectorizer(
#     max_features=30000,
#     ngram_range=(1, 2),
#     min_df=3,
#     max_df=0.9,
#     sublinear_tf=True,
#     stop_words=None,
#     analyzer='word'
# )
# # tfidf_name = TfidfVectorizer(
# #     max_features=5000,
# #     ngram_range=(1, 2),
# #     min_df=1,
# #     max_df=0.95,
# #     sublinear_tf=True,
# #     stop_words=None,
# #     analyzer='word'
# # )

# # Clean text just to be safe
# # train["Item_Name"] = train["Item_Name"].fillna("").astype(str)
# # test["Item_Name"] = test["Item_Name"].fillna("").astype(str)

# train["catalog_content"] = train["catalog_content"].fillna("").astype(str)
# test["catalog_content"] = test["catalog_content"].fillna("").astype(str)

# tfidf_train = tfidf.fit_transform(train["catalog_content"])
# tfidf_test = tfidf.transform(test["catalog_content"])
# # Item_train = tfidf_name.fit_transform(train["Item_Name"])
# # Item_test = tfidf_name.transform(test["Item_Name"])

In [14]:
for col in ["item_name", "bullet_points", "product_desc"]:
    train[col] = train[col].fillna("").astype(str)
    test[col] = test[col].fillna("").astype(str)
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_params = dict(
    max_features=30000,
    ngram_range=(1,2),
    min_df=3,
    max_df=0.9,
    sublinear_tf=True,
    analyzer='word'
)

# Item Name
tfidf_item = TfidfVectorizer(**tfidf_params)
tfidf_item_train = tfidf_item.fit_transform(train["item_name"])
tfidf_item_test  = tfidf_item.transform(test["item_name"])

# Bullet Points
tfidf_bullet = TfidfVectorizer(**tfidf_params)
tfidf_bullet_train = tfidf_bullet.fit_transform(train["bullet_points"])
tfidf_bullet_test  = tfidf_bullet.transform(test["bullet_points"])

# Product Description
tfidf_desc = TfidfVectorizer(**tfidf_params)
tfidf_desc_train = tfidf_desc.fit_transform(train["product_desc"])
tfidf_desc_test  = tfidf_desc.transform(test["product_desc"])

from scipy.sparse import hstack

X_train_tfidf = hstack([tfidf_item_train, tfidf_bullet_train, tfidf_desc_train])
X_test_tfidf  = hstack([tfidf_item_test, tfidf_bullet_test, tfidf_desc_test])


Combine structured features:

In [15]:
structured_features = ["log_value",
    "bullet_point_count",
    "word_count_item",
    "word_count_bullet",
    "word_count_desc",
    "word_count_total",
    "char_count_total",
    "digit_ratio_total",
    "avg_word_len_total",
    "unit_enc"] + [f"has_{kw}" for kw in keywords]
scaler = StandardScaler()
train_struct = scaler.fit_transform(train[structured_features])
test_struct = scaler.transform(test[structured_features])

X = hstack([X_train_tfidf, train_struct])
X = X.tocsr()
X_test = hstack([X_test_tfidf, test_struct])
X_test = X_test.tocsr()
y = np.log1p(train["price"].values)

In [16]:
# train.head(10)
# # train.columns

LightGBM Training:

In [17]:
# Define SMAPE function
def smape(y_true, y_pred):
    denominator = (np.abs(y_true) + np.abs(y_pred) + 0.000001) / 2
    diff = np.abs(y_true - y_pred) / denominator
    diff[denominator == 0] = 0  # avoid division by zero
    return 100 * np.mean(diff)

# K-Fold parameters
params = {
    "objective": "regression",
    "metric": "mae",  # still using MAE internally
    "learning_rate": 0.05,
    "num_leaves": 63,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "random_state": 22,
}

kf = KFold(n_splits=5, shuffle=True, random_state=22)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))
best_iterations = []

for fold, (trn_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"\n----- Fold {fold+1} -----")

    X_tr, X_val = X[trn_idx], X[val_idx]
    y_tr, y_val = y[trn_idx], y[val_idx]

    train_set = lgb.Dataset(X_tr, label=y_tr)
    val_set = lgb.Dataset(X_val, label=y_val, reference=train_set)

    model = lgb.train(
        params,
        train_set,
        num_boost_round=2000,
        valid_sets=[train_set, val_set],
        valid_names=["train", "val"],
        callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(100)],
    )
    best_iterations.append(model.best_iteration)

    oof_preds[val_idx] = model.predict(X_val, num_iteration=model.best_iteration)
    test_preds += model.predict(X_test, num_iteration=model.best_iteration) / kf.n_splits

    # Compute SMAPE for this fold
    fold_smape = smape(np.expm1(y_val), np.expm1(oof_preds[val_idx]))
    print(f"Fold {fold+1} SMAPE: {fold_smape:.4f}")

# Overall CV SMAPE
overall_smape = smape(np.expm1(y), np.expm1(oof_preds))
print("\nOverall CV SMAPE:", overall_smape)
average_best_iteration = int(np.mean(best_iterations))


----- Fold 1 -----
Training until validation scores don't improve for 100 rounds
[100]	train's l1: 0.515691	val's l1: 0.545055
[200]	train's l1: 0.471164	val's l1: 0.523761
[300]	train's l1: 0.442325	val's l1: 0.514657
[400]	train's l1: 0.420068	val's l1: 0.509882
[500]	train's l1: 0.401082	val's l1: 0.50742
[600]	train's l1: 0.384915	val's l1: 0.505485
[700]	train's l1: 0.370682	val's l1: 0.504158
[800]	train's l1: 0.357033	val's l1: 0.503754
[900]	train's l1: 0.345058	val's l1: 0.503523
Early stopping, best iteration is:
[893]	train's l1: 0.345795	val's l1: 0.503478
Fold 1 SMAPE: 50.8282

----- Fold 2 -----
Training until validation scores don't improve for 100 rounds
[100]	train's l1: 0.513576	val's l1: 0.551934
[200]	train's l1: 0.468672	val's l1: 0.530766
[300]	train's l1: 0.439973	val's l1: 0.522917
[400]	train's l1: 0.417371	val's l1: 0.517593
[500]	train's l1: 0.398979	val's l1: 0.514224
[600]	train's l1: 0.382408	val's l1: 0.512043
[700]	train's l1: 0.368308	val's l1: 0.51086

Final Model Training:

In [18]:
# Prepare full dataset
full_train_set = lgb.Dataset(X, label=y)

# Train final model on all data
final_model = lgb.train(
    params,
    full_train_set,
    num_boost_round=average_best_iteration
)

Final model prediction:

In [19]:
final_preds_log = final_model.predict(X_test, num_iteration=final_model.best_iteration)
final_preds = np.expm1(final_preds_log)
print(final_preds)

[14.12091743 15.52790163 17.55879445 ...  4.10661843 12.70322145
 11.04640486]


Code for getting data for ensembling:

In [23]:
kf = KFold(n_splits=5, shuffle=True, random_state=22)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))
best_iterations = []

# --- 4. Your existing K-Fold Training and Prediction Loop ---
# This loop is identical to the one you provided.
print("Starting LightGBM K-Fold Training...")
for fold, (trn_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"\n----- Fold {fold+1} -----")

    X_tr, X_val = X[trn_idx], X[val_idx]
    y_tr, y_val = y[trn_idx], y[val_idx]

    train_set = lgb.Dataset(X_tr, label=y_tr)
    val_set = lgb.Dataset(X_val, label=y_val, reference=train_set)

    model = lgb.train(
        params,
        train_set,
        num_boost_round=2000,
        valid_sets=[train_set, val_set],
        valid_names=["train", "val"],
        callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(100)],
    )
    best_iterations.append(model.best_iteration)

    oof_preds[val_idx] = model.predict(X_val, num_iteration=model.best_iteration)
    test_preds += model.predict(X_test, num_iteration=model.best_iteration) / kf.n_splits

print("\nK-Fold training complete. Saving predictions...")

# Save the out-of-fold predictions for the training set
oof_df = pd.DataFrame({'sample_id': train['sample_id'], 'lgbm_oof_preds': np.expm1(oof_preds)})
oof_df.to_csv('lgbm_oof_preds.csv', index=False)

# Save the averaged predictions for the test set
test_preds_df = pd.DataFrame({'sample_id': test['sample_id'], 'lgbm_test_preds': np.expm1(test_preds)})
test_preds_df.to_csv('lgbm_test_preds.csv', index=False)

print("OOF and test predictions saved successfully.")
print(f"OOF predictions shape: {oof_df.shape}")
print(f"Test predictions shape: {test_preds_df.shape}")

Starting LightGBM K-Fold Training...

----- Fold 1 -----
Training until validation scores don't improve for 100 rounds
[100]	train's l1: 0.515691	val's l1: 0.545055


KeyboardInterrupt: 

Save predictions:

In [24]:
# Save the out-of-fold predictions for the training set
# oof_df = pd.DataFrame({'sample_id': train['sample_id'], 'lgbm_oof_preds': np.expm1(oof_preds)})
oof_df['lgbm_oof_preds']=np.expm1(oof_preds)
oof_df.to_csv('lgbm_oof_preds.csv', index=False)

# Save the averaged predictions for the test set
# test_preds_df = pd.DataFrame({'sample_id': test['sample_id'], 'lgbm_test_preds': np.expm1(test_preds)})
test_preds_df['lgbm_oof_preds']=np.expm1(test_preds)

test_preds_df.to_csv('lgbm_test_preds.csv', index=False)

print("OOF and test predictions saved successfully.")
print(f"OOF predictions shape: {oof_df.shape}")
print(f"Test predictions shape: {test_preds_df.shape}")

OOF and test predictions saved successfully.
OOF predictions shape: (75000, 2)
Test predictions shape: (75000, 3)


In [25]:
test_preds_df['lgbm_oof_preds']

0        0.0
1        0.0
2        0.0
3        0.0
4        0.0
        ... 
74995    0.0
74996    0.0
74997    0.0
74998    0.0
74999    0.0
Name: lgbm_oof_preds, Length: 75000, dtype: float64

In [21]:
submission = pd.DataFrame({
    "sample_id": test["sample_id"],
    "price": final_preds
})

submission.to_csv(r"predictions_lgbm.csv", index=False)
print(submission.head())

   sample_id      price
0     100179  14.120917
1     245611  15.527902
2     146263  17.558794
3      95658   9.588309
4      36806  34.440138


In [22]:
import joblib
joblib.dump(final_model, "final_lgbm_model.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")
joblib.dump(le, "label_encoder.pkl")

NameError: name 'tfidf' is not defined

# EfficientNet Image Features Integration

Now we'll load and integrate the EfficientNet-generated image features to enhance our LightGBM model with visual information.

In [ ]:
# Load and examine EfficientNet image features
print("Loading EfficientNet image features...")

try:
    # Load the training image features
    image_features_train = np.load('train_resnet50_features_efficientnet1.npy')
    print(f"✓ Loaded training image features: {image_features_train.shape}")
    
    # Check if we have test image features
    try:
        image_features_test = np.load('test_resnet50_features_efficientnet1.npy')
        print(f"✓ Loaded test image features: {image_features_test.shape}")
    except FileNotFoundError:
        print("⚠️  Test image features not found. You'll need to generate them using the same EfficientNet model.")
        image_features_test = None
        
    # Display feature statistics
    print(f"\nImage Feature Statistics:")
    print(f"Training features shape: {image_features_train.shape}")
    print(f"Feature range: {image_features_train.min():.4f} to {image_features_train.max():.4f}")
    print(f"Feature mean: {image_features_train.mean():.4f}")
    print(f"Feature std: {image_features_train.std():.4f}")
    
    # Check for NaN values
    nan_count = np.isnan(image_features_train).sum()
    print(f"NaN values in training features: {nan_count}")
    
    if image_features_test is not None:
        nan_count_test = np.isnan(image_features_test).sum()
        print(f"NaN values in test features: {nan_count_test}")
    
except FileNotFoundError as e:
    print(f"✗ Error loading image features: {e}")
    print("Please ensure the .npy files are in the current directory.")
    image_features_train = None
    image_features_test = None

In [ ]:
# Preprocess and validate image features
if image_features_train is not None:
    
    # Check if dimensions match our training data
    print(f"\nValidating feature alignment:")
    print(f"Training data samples: {len(train)}")
    print(f"Image features samples: {image_features_train.shape[0]}")
    
    if len(train) != image_features_train.shape[0]:
        print("⚠️  Warning: Mismatch between training data and image features!")
        # You may need to handle this by matching sample_ids
        min_samples = min(len(train), image_features_train.shape[0])
        print(f"Using first {min_samples} samples for both")
        train_subset = train.iloc[:min_samples].copy()
        image_features_train = image_features_train[:min_samples]
    else:
        train_subset = train.copy()
    
    # Handle NaN values in image features
    if np.isnan(image_features_train).any():
        print("Filling NaN values in image features with mean...")
        from sklearn.impute import SimpleImputer
        imputer = SimpleImputer(strategy='mean')
        image_features_train = imputer.fit_transform(image_features_train)
        
        if image_features_test is not None:
            image_features_test = imputer.transform(image_features_test)
    
    # Standardize image features
    from sklearn.preprocessing import StandardScaler
    print("Standardizing image features...")
    image_scaler = StandardScaler()
    image_features_train_scaled = image_scaler.fit_transform(image_features_train)
    
    if image_features_test is not None:
        image_features_test_scaled = image_scaler.transform(image_features_test)
    else:
        image_features_test_scaled = None
    
    print(f"✓ Image features preprocessed and scaled")
    print(f"Scaled feature range: {image_features_train_scaled.min():.4f} to {image_features_train_scaled.max():.4f}")
    
else:
    print("Skipping image feature processing - features not available")
    image_features_train_scaled = None
    image_features_test_scaled = None

In [ ]:
# Combine image features with existing features
if image_features_train_scaled is not None:
    
    print("Combining text/structured features with image features...")
    
    # Get the existing feature matrices (using the same preprocessing as before)
    # Note: We need to recompute X and X_test with the correct subset if needed
    if 'train_subset' in locals():
        # Recompute features for the subset
        y_subset = np.log1p(train_subset["price"].values)
        
        # Get existing features for subset
        X_subset = X[:len(train_subset)]  # Use the pre-computed X matrix
        
    else:
        X_subset = X
        y_subset = y
    
    # Combine sparse text features with dense image features
    print(f"Text/structured features shape: {X_subset.shape}")
    print(f"Image features shape: {image_features_train_scaled.shape}")
    
    # Convert image features to sparse matrix for efficient combination
    from scipy.sparse import csr_matrix
    image_features_sparse = csr_matrix(image_features_train_scaled)
    
    # Combine all features
    X_combined = hstack([X_subset, image_features_sparse])
    X_combined = X_combined.tocsr()
    
    print(f"Combined features shape: {X_combined.shape}")
    
    # Prepare test features if available
    if image_features_test_scaled is not None:
        print("Preparing combined test features...")
        image_features_test_sparse = csr_matrix(image_features_test_scaled)
        X_test_combined = hstack([X_test, image_features_test_sparse])
        X_test_combined = X_test_combined.tocsr()
        print(f"Combined test features shape: {X_test_combined.shape}")
    else:
        print("⚠️  Test image features not available - using text features only for test set")
        X_test_combined = X_test
    
    # Update feature names for tracking
    image_feature_names = [f"efficientnet_feature_{i}" for i in range(image_features_train_scaled.shape[1])]
    
    print(f"✓ Successfully combined features!")
    print(f"  - Text/structured features: {X_subset.shape[1]}")
    print(f"  - Image features: {image_features_train_scaled.shape[1]}")
    print(f"  - Total combined features: {X_combined.shape[1]}")
    
else:
    print("Using original features without image enhancement")
    X_combined = X
    X_test_combined = X_test
    y_subset = y

# Enhanced LightGBM Training with Image Features

Now we'll train an enhanced LightGBM model using both text/structured features and EfficientNet image features.

In [ ]:
# Enhanced LightGBM K-Fold Training with Image Features
print("Starting Enhanced LightGBM Training with Image Features...")

# Enhanced parameters for multimodal features
enhanced_params = {
    "objective": "regression",
    "metric": "mae",
    "learning_rate": 0.05,
    "num_leaves": 127,  # Increased for more complex features
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "min_data_in_leaf": 20,
    "lambda_l1": 0.1,
    "lambda_l2": 0.1,
    "verbose": -1,
    "random_state": 42,
}

# Initialize cross-validation
kf_enhanced = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds_enhanced = np.zeros(len(y_subset))
test_preds_enhanced = np.zeros(len(X_test_combined) if image_features_test_scaled is not None else len(test))
best_iterations_enhanced = []

for fold, (trn_idx, val_idx) in enumerate(kf_enhanced.split(X_combined, y_subset)):
    print(f"\n----- Enhanced Fold {fold+1} -----")
    
    X_tr, X_val = X_combined[trn_idx], X_combined[val_idx]
    y_tr, y_val = y_subset[trn_idx], y_subset[val_idx]
    
    train_set = lgb.Dataset(X_tr, label=y_tr)
    val_set = lgb.Dataset(X_val, label=y_val, reference=train_set)
    
    model_enhanced = lgb.train(
        enhanced_params,
        train_set,
        num_boost_round=2000,
        valid_sets=[train_set, val_set],
        valid_names=["train", "val"],
        callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(100)],
    )
    best_iterations_enhanced.append(model_enhanced.best_iteration)
    
    # Predictions
    oof_preds_enhanced[val_idx] = model_enhanced.predict(X_val, num_iteration=model_enhanced.best_iteration)
    test_preds_enhanced += model_enhanced.predict(X_test_combined, num_iteration=model_enhanced.best_iteration) / kf_enhanced.n_splits
    
    # Compute SMAPE for this fold
    fold_smape = smape(np.expm1(y_val), np.expm1(oof_preds_enhanced[val_idx]))
    print(f"Enhanced Fold {fold+1} SMAPE: {fold_smape:.4f}")

# Overall enhanced CV SMAPE
overall_smape_enhanced = smape(np.expm1(y_subset), np.expm1(oof_preds_enhanced))
print(f"\n✅ Enhanced Model Overall CV SMAPE: {overall_smape_enhanced:.4f}")

# Compare with original model performance if available
if 'overall_smape' in locals():
    improvement = overall_smape - overall_smape_enhanced
    print(f"📈 Improvement over text-only model: {improvement:.4f} SMAPE points")
    print(f"   Original SMAPE: {overall_smape:.4f}")
    print(f"   Enhanced SMAPE: {overall_smape_enhanced:.4f}")

average_best_iteration_enhanced = int(np.mean(best_iterations_enhanced))
print(f"Average best iteration: {average_best_iteration_enhanced}")

In [ ]:
# Train Final Enhanced Model and Analyze Feature Importance
print("Training final enhanced model on full dataset...")

# Train final enhanced model
full_train_set_enhanced = lgb.Dataset(X_combined, label=y_subset)
final_model_enhanced = lgb.train(
    enhanced_params,
    full_train_set_enhanced,
    num_boost_round=average_best_iteration_enhanced
)

# Generate final predictions
final_preds_log_enhanced = final_model_enhanced.predict(X_test_combined, num_iteration=final_model_enhanced.best_iteration)
final_preds_enhanced = np.expm1(final_preds_log_enhanced)

print(f"✅ Enhanced model training complete!")
print(f"Enhanced predictions range: ${final_preds_enhanced.min():.2f} to ${final_preds_enhanced.max():.2f}")

# Analyze feature importance
print(f"\n📊 FEATURE IMPORTANCE ANALYSIS")
print("=" * 50)

feature_importance = final_model_enhanced.feature_importance(importance_type='gain')
feature_names_combined = list(range(X_combined.shape[1]))  # Generic names since we don't have exact mapping

# Get top important features
importance_df = pd.DataFrame({
    'feature_idx': feature_names_combined,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

print("Top 20 Most Important Features:")
for i, (idx, importance) in enumerate(importance_df.head(20).values):
    feature_type = "Image" if idx >= (X_combined.shape[1] - image_features_train_scaled.shape[1]) else "Text/Structured"
    print(f"{i+1:2d}. Feature {idx:<4} ({feature_type:<12}): {importance:>8.0f}")

# Calculate contribution of image features
if image_features_train_scaled is not None:
    text_features_count = X_combined.shape[1] - image_features_train_scaled.shape[1]
    image_importance_sum = importance_df[importance_df['feature_idx'] >= text_features_count]['importance'].sum()
    total_importance = importance_df['importance'].sum()
    image_contribution = (image_importance_sum / total_importance) * 100
    
    print(f"\n📈 Image Features Contribution: {image_contribution:.1f}%")
    print(f"Text/Structured Features Contribution: {100-image_contribution:.1f}%")

In [ ]:
# Save Enhanced Predictions for Ensembling
print("Saving enhanced predictions for ensembling...")

# Determine the sample IDs to use
if 'train_subset' in locals():
    train_sample_ids = train_subset['sample_id']
    test_sample_ids = test['sample_id']
else:
    train_sample_ids = train['sample_id']  
    test_sample_ids = test['sample_id']

# Save enhanced out-of-fold predictions
oof_enhanced_df = pd.DataFrame({
    'sample_id': train_sample_ids,
    'lgbm_enhanced_oof_preds': np.expm1(oof_preds_enhanced)
})
oof_enhanced_df.to_csv('lgbm_enhanced_oof_preds.csv', index=False)

# Save enhanced test predictions
test_enhanced_df = pd.DataFrame({
    'sample_id': test_sample_ids,
    'lgbm_enhanced_test_preds': np.expm1(test_preds_enhanced)
})
test_enhanced_df.to_csv('lgbm_enhanced_test_preds.csv', index=False)

# Save final enhanced submission
submission_enhanced = pd.DataFrame({
    "sample_id": test_sample_ids,
    "price": final_preds_enhanced
})
submission_enhanced.to_csv("predictions_lgbm_enhanced.csv", index=False)

print("✅ Enhanced predictions saved:")
print(f"  - lgbm_enhanced_oof_preds.csv ({len(oof_enhanced_df)} rows)")
print(f"  - lgbm_enhanced_test_preds.csv ({len(test_enhanced_df)} rows)")
print(f"  - predictions_lgbm_enhanced.csv ({len(submission_enhanced)} rows)")

# Save enhanced model components
import joblib

joblib.dump(final_model_enhanced, "final_lgbm_enhanced_model.pkl")
if image_features_train_scaled is not None:
    joblib.dump(image_scaler, "image_feature_scaler.pkl")

print("✅ Enhanced model components saved:")
print("  - final_lgbm_enhanced_model.pkl")
if image_features_train_scaled is not None:
    print("  - image_feature_scaler.pkl")

# Performance summary
print(f"\n🎯 ENHANCED MODEL PERFORMANCE SUMMARY")
print("=" * 50)
print(f"Enhanced Model CV SMAPE: {overall_smape_enhanced:.4f}")
if 'overall_smape' in locals():
    print(f"Original Model CV SMAPE: {overall_smape:.4f}")
    print(f"Improvement: {overall_smape - overall_smape_enhanced:+.4f}")
print(f"Total Features Used: {X_combined.shape[1]:,}")
if image_features_train_scaled is not None:
    print(f"  - Text/Structured: {X_combined.shape[1] - image_features_train_scaled.shape[1]:,}")
    print(f"  - Image Features: {image_features_train_scaled.shape[1]:,}")
    print(f"  - Image Contribution: {image_contribution:.1f}%")